<a href="https://colab.research.google.com/github/heetaamin/ml-assignment2/blob/main/final/kNN_Notebook1_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 - kNN, global preprocessing

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic.csv'
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

Mounted at /content/drive
Shape: (257673, 44)


In [2]:
# drop structurally redundant columns
# id is unique per row - keeping it just gives the tree room to overfit
# ct_ftp_cmd/sloss/dloss are near-duplicates of another retained feature and tcprtt is just synack + ackdat added together.

df_knn = df.copy()
df_knn = df_knn.drop(columns=['id', 'ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt'])
print("Shape:", df_knn.shape)

Shape: (257673, 39)


In [3]:
# fix the two invalid-value issues from EDA

# state has one row = 'no', not in the documented categories - recode to the mode (FIN).

# is_ftp_login has 26 rows with 2 or 4 instead of 0/1. All 26 have a nonzero ct_ftp_cmd too, so a login must have happened - map to 1.

state_mode = df_knn['state'].mode()[0]
df_knn['state'] = df_knn['state'].replace('no', state_mode)

df_knn['is_ftp_login'] = df_knn['is_ftp_login'].replace({2: 1, 4: 1})

print("state fixed:", not (df_knn['state'] == 'no').any())
print("is_ftp_login fixed:", sorted(df_knn['is_ftp_login'].unique()))

state fixed: True
is_ftp_login fixed: [np.int64(0), np.int64(1)]


In [4]:
# service missing-value flag

df_knn['service_missing'] = (df_knn['service'] == '?').astype(int)
print("Missing flagged:", df_knn['service_missing'].sum())

Missing flagged: 141321


In [5]:
# The seven features formed a highly redundant connection-count cluster, with seven pairwise correlations exceeding 0.9.
# To reduce redundancy, ct_srv_src and ct_srv_dst were retained as source-side and destination-side representatives respectively.
# A separate cross-validation comparison of the full seven-feature cluster vs the two-feature subset provided supporting diagnostic evidence for the reduction

cluster_drop = ['ct_dst_src_ltm', 'ct_src_dport_ltm', 'ct_dst_ltm',
                 'ct_src_ltm', 'ct_dst_sport_ltm']
df_knn = df_knn.drop(columns=cluster_drop)
print("Shape:", df_knn.shape)

Shape: (257673, 35)


In [6]:
# Check duplicates on raw data

n_dupes = df_knn.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df_knn)*100:.2f}%)")

Duplicate rows: 107430 (41.69%)


In [7]:
# Remove duplicate rows

print("Shape before dedup:", df_knn.shape)
df_knn = df_knn.drop_duplicates()
print("Shape after dedup:", df_knn.shape)

Shape before dedup: (257673, 35)
Shape after dedup: (150243, 35)


In [8]:
# Save the semi-raw dataset

df_knn.to_csv('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_knn_semiraw.csv', index=False)
print("Saved. Shape:", df_knn.shape)

Saved. Shape: (150243, 35)


In [9]:
# alternate semi-raw dataset keeping all 7 connection-count cluster features

df_knn_7cluster = df.copy()
df_knn_7cluster = df_knn_7cluster.drop(columns=['id', 'ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt'])

state_mode = df_knn_7cluster['state'].mode()[0]
df_knn_7cluster['state'] = df_knn_7cluster['state'].replace('no', state_mode)
df_knn_7cluster['is_ftp_login'] = df_knn_7cluster['is_ftp_login'].replace({2: 1, 4: 1})
df_knn_7cluster['service_missing'] = (df_knn_7cluster['service'] == '?').astype(int)

print("Shape before dedup (7-cluster branch):", df_knn_7cluster.shape)
n_dupes = df_knn_7cluster.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df_knn_7cluster)*100:.2f}%)")

df_knn_7cluster = df_knn_7cluster.drop_duplicates()
print("Shape after dedup:", df_knn_7cluster.shape)

df_knn_7cluster.to_csv('/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 2/networkTraffic_knn_semiraw_7cluster.csv', index=False)
print("Saved.")

Shape before dedup (7-cluster branch): (257673, 40)
Duplicate rows: 94928 (36.84%)
Shape after dedup: (162745, 40)
Saved.
